# Benchmark preservation checks

This local check compares regenerated figures with the previously registered paper release. It ignores only SVG creation dates and compares PNG pixels exactly. It also checks the smoke wrapper command-line help from an independent working directory; the evidence notebook separately executes the wrapper from the notebooks directory.

Publication note: author path labels were normalized. Saved numerical outputs record the earlier execution. This source record needs the author archives; use the current portable evidence and preservation notebooks for checkout-level verification.

In [1]:
from pathlib import Path
import hashlib, importlib.metadata, json, platform, sys
import numpy as np
import pandas as pd
from IPython.display import display

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1048576), b''):
            digest.update(block)
    return digest.hexdigest()

HISTORICAL_MAC_ENVIRONMENT = {
    'manager': 'existing micromamba-compatible environment, reused without modification',
    'name': 'chronoclone',
    'absolute_macbook_path': '/path/to/ChronoClone/.runtime/chronoclone',
    'kernel': 'chronoclone',
}
runtime = dict(HISTORICAL_MAC_ENVIRONMENT,
    current_prefix=sys.prefix, current_executable=sys.executable, python=platform.python_version(),
    versions={name: importlib.metadata.version(name) for name in
              ['numpy','pandas','scipy','scikit-learn','scanpy','matplotlib','nbformat','nbclient']})

PROJECT=Path('/path/to/ChronoClone')
BENCHMARKS=PROJECT/'analysis_workspace/ChronoClone_paper/benchmarks'
OLD=PROJECT/'analysis_workspace/releases/chronoclone_paper_repository_20260908/repository/analyses/01_ode_benchmark'
runtime['execution_command']='python benchmarks/run_notebook.py benchmarks/curation/0002_preservation_checks.ipynb --kernel chronoclone'
display(pd.DataFrame([runtime]).T)


,0
manager,"existing micromamba-compatible environment, re..."
name,chronoclone
absolute_macbook_path,~/.reference-science/conda/envs/...
kernel,chronoclone
current_prefix,~/.reference-science/conda/envs/...
current_executable,~/.reference-science/conda/envs/...
python,3.11.15
versions,"{'numpy': '2.4.6', 'pandas': '2.3.3', 'scipy':..."
execution_command,python benchmarks/run_notebook.py benchmarks/c...


## Compare all regenerated figure contents

In [2]:
import re
from PIL import Image
rows=[]
for path in sorted((BENCHMARKS/'ode_noise/figures').rglob('*.svg')):
    relative=path.relative_to(BENCHMARKS/'ode_noise/figures')
    old=OLD/'figures'/relative
    normalized=re.sub(r'<dc:date>.*?</dc:date>','',path.read_text())
    old_normalized=re.sub(r'<dc:date>.*?</dc:date>','',old.read_text())
    rows.append({'figure':str(relative),'comparison':'SVG content excluding creation date','passed':normalized==old_normalized})
for path in sorted((BENCHMARKS/'ode_noise/figures/manuscript_four_method').glob('*.png')):
    old=OLD/'figures/manuscript_four_method'/path.name
    rows.append({'figure':path.name,'comparison':'PNG pixels','passed':np.array_equal(np.asarray(Image.open(path)),np.asarray(Image.open(old)))})
frame=pd.DataFrame(rows)
assert frame.passed.all(),frame.loc[~frame.passed]
frame.to_csv(BENCHMARKS/'curation/figure_preservation.csv',index=False)
display(frame.groupby('comparison').agg(files=('figure','size'),passed=('passed','sum')))
print('Visual spot checks: Gaussian ARI and silhouette-coverage PNGs inspected individually; no clipping or overlap identified. All remaining regenerated content matches its preserved source.')


,files,passed
comparison,,
PNG pixels,14,14
SVG content excluding creation date,29,29


Visual spot checks: Gaussian ARI and silhouette-coverage PNGs inspected individually; no clipping or overlap identified. All remaining regenerated content matches its preserved source.


## Confirm notebook execution records and independent working directory

In [3]:
import nbformat, subprocess, tempfile
records=[]
for name in ['0003_t1_locked_method_noise_results.ipynb','0005_t1_locked_four_method_noise_results.ipynb','0006_benchmark_evidence_and_smoke.ipynb']:
    path=BENCHMARKS/'ode_noise/notebooks'/name
    notebook=nbformat.read(path,as_version=4)
    cells=[cell for cell in notebook.cells if cell.cell_type=='code']
    assert [cell.execution_count for cell in cells]==list(range(1,len(cells)+1))
    assert not [output for cell in cells for output in cell.outputs if output.output_type=='error']
    records.append({'notebook':name,'code_cells':len(cells),'retained_output_cells':sum(bool(c.outputs) for c in cells),'sha256':sha256(path)})
# Help resolves the script and parser without fitting another condition.
with tempfile.TemporaryDirectory(prefix='cc_paper_cwd_') as temporary:
    result=subprocess.run([sys.executable,str(BENCHMARKS/'ode_noise/scripts/run_smoke.py'),'--help'],cwd=temporary,capture_output=True,text=True)
    assert result.returncode==0,result.stderr
manifest={'status':'pass','figure_comparisons':len(rows),'notebooks':records,'smoke_help_independent_cwd':True,'runtime':runtime}
(BENCHMARKS/'curation/preservation_validation.json').write_text(json.dumps(manifest,indent=2)+'\n')
display(pd.DataFrame(records))


,notebook,code_cells,retained_output_cells,sha256
0,0003_t1_locked_method_noise_results.ipynb,20,20,650e4538109d5d888a2bbee447420bfcb9b771f3f0f063...
1,0005_t1_locked_four_method_noise_results.ipynb,19,19,e968e86f04a2424ed2a797f2ebb461c8dc95e22434a1cd...
2,0006_benchmark_evidence_and_smoke.ipynb,5,5,eda61b342f1f485cf2a67591e02fa5cf0556d258d39fdd...
